In [ ]:
import os, glob, numpy as np, torch
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from torchvision.datasets.utils import download_and_extract_archive

import importlib
import train

importlib.reload(train)

from models import resnet
from train import entrainer

# RESNET sur Tiny ImageNet 

(train : 100 000 images 64x64x3, val : 10 000 images, 200 classes, pas de test)

## Chargement des données

In [ ]:
URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

def _load(path):
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)

def _load_many(paths):
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(_load, paths)))

def load_tiny_imagenet(data_dir="./data"):
    """Télécharge Tiny ImageNet si besoin et renvoie 4 tableaux numpy uint8/int64 :
    x_train, y_train, x_val, y_val."""
    cache = os.path.join(data_dir, "tiny_imagenet_cache.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return d["x_tr"], d["y_tr"], d["x_va"], d["y_va"]

    root = os.path.join(data_dir, "tiny-imagenet-200")
    if not os.path.isdir(root):
        download_and_extract_archive(URL, data_dir)   # ~240 Mo

    wnids = [l.strip() for l in open(f"{root}/wnids.txt")]
    c2i = {w: i for i, w in enumerate(wnids)}

    tr_paths, tr_lab = [], []
    for w in wnids:
        ps = sorted(glob.glob(f"{root}/train/{w}/images/*.JPEG"))
        tr_paths += ps
        tr_lab += [c2i[w]] * len(ps)

    va_paths, va_lab = [], []
    for line in open(f"{root}/val/val_annotations.txt"):
        img, w = line.split("\t")[:2]
        va_paths.append(f"{root}/val/images/{img}")
        va_lab.append(c2i[w])

    x_tr, y_tr = _load_many(tr_paths), np.array(tr_lab, dtype=np.int64)
    x_va, y_va = _load_many(va_paths), np.array(va_lab, dtype=np.int64)
    np.savez(cache, x_tr=x_tr, y_tr=y_tr, x_va=x_va, y_va=y_va)
    return x_tr, y_tr, x_va, y_va


In [ ]:
x_tr, y_tr, x_va, y_va = load_tiny_imagenet("./data")

def to_t(x, y):
    x = torch.from_numpy(x).permute(0, 3, 1, 2).contiguous() 
    y = torch.from_numpy(y)                               
    return x, y

Xtr, ytr = to_t(x_tr, y_tr)
Xva, yva = to_t(x_va, y_va)
del x_tr, y_tr, x_va, y_va

mean = (Xtr.float() / 255).mean(dim=(0, 2, 3), keepdim=True)
std = (Xtr.float() / 255).std(dim=(0, 2, 3), keepdim=True)

data = {
    "Xtr": Xtr, "ytr": ytr, "Xva": Xva, "yva": yva,
    "prep": lambda x: (x.float() / 255 - mean) / std,
}

In [ ]:
torch.manual_seed(0)
model, opt = resnet(output_shape=200, widths=(8, 16, 32), blocs=(1, 1, 1), sd_max=0.5, in_channels=3)

print("paramètres :", sum(p.numel() for p in model.parameters()))

In [ ]:
df = entrainer(model, opt, data, epochs=6, t_early=3, p=0.1, batch=128)
print(df[["epoch", "scale", "train_loss", "train_acc", "val_loss", "val_acc"]].round(3))